# Notebook 9 — Inferência Econométrica de Desempenho e Caracterização do Benchmark

**TCC — Pedro Augusto Pinheiro Reis · Ciências Contábeis · UFG**

Este notebook operacionaliza a caracterização econométrica do benchmark (IBOVESPA) e executa os testes de hipóteses estatísticas comparando as carteiras ótimas contra o mercado.

## 1. Importações e Configurações

In [1]:
import sys
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.stats.diagnostic import het_arch
from arch import arch_model
from arch.unitroot import VarianceRatio

from utils.config_loader import carregar_parametros
from utils.inferencia import (
    fmt_pvalor,
    sharpe,
    sortino,
    sharpe_de_excesso,
    sortino_de_excesso,
    cagr,
    max_drawdown,
    _wald_spanning,
    _jk_memmel,
    bootstrap_ic,
    diagnostico_residuos,
    diagnosticos_serie,
    lw_bootstrap_sharpe,
    lw_bootstrap_sortino,
    TRADING_DAYS
)
from statsmodels.stats.multitest import multipletests

warnings.filterwarnings("ignore")

# ==============================================================================
# CONFIGURAÇÃO METODOLÓGICA GLOBAL (REAMOSTRAGEM - SEÇÃO 3.7.2)
# ==============================================================================
SEED = 42
BOOTSTRAP_REPS = 2000
BOOTSTRAP_BLOCK_MEAN = 10
ALPHA_SIG = 0.05
# ==============================================================================

cfg = carregar_parametros()
SPANNING_MAX_LAGS = cfg.get("SPANNING_MAX_LAGS", 5)

project_root = Path.cwd().parent.parent
INPUT_DIR_IBOV     = project_root / "data" / "Ibov" / "Ibov_Diario"
INPUT_DIR_CDI      = project_root / "data" / "CDI"
INPUT_DIR_STRATEGY = project_root / "data" / "Estrategias"
OUTPUT_DIR         = project_root / "data" / "Estrategias"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ESTRATEGIAS = [
    "EqualWeight", "EqualWeight_BuyHold", "InvVol",
    "MinVar", "MinVar_c10", "MaxSharpe", "MaxSharpe_c10",
    "MaxOmega", "MaxSortino", "MaxKappa3", "MinCVaR", "MinCDaR",
    "BL_classico", "BL_classico_c10", "BL_downside", "BL_downside_c10"
]

BENCHMARK  = "EqualWeight"   # DeMiguel et al. (2009) — desafiante canônico 1/N

print(f"✓ Configurações carregadas. Total de estratégias: {len(ESTRATEGIAS)} | benchmark: {BENCHMARK}")
print(f"  Inclui EqualWeight_BuyHold (1/N sem rebalanceamento) para análise de concentração.")


✓ Configurações carregadas. Total de estratégias: 16 | benchmark: EqualWeight
  Inclui EqualWeight_BuyHold (1/N sem rebalanceamento) para análise de concentração.


## 2. Ingestão de Dados e Alinhamento do Benchmark

In [2]:
ibov_path = INPUT_DIR_IBOV / "ibov_diario_2010_2026.csv"
if not ibov_path.exists():
    raise FileNotFoundError("ibov_diario_2010_2026.csv não encontrado.")
ibov = pd.read_csv(ibov_path, parse_dates=["data"])
ibov = ibov.dropna(subset=["ibov_ret_simples"]).reset_index(drop=True)

cdi_path = INPUT_DIR_CDI / "cdi_diario_bcb_2010_atual.csv"
if not cdi_path.exists():
    raise FileNotFoundError("cdi_diario_bcb_2010_atual.csv não encontrado.")
cdi_df = pd.read_csv(cdi_path, sep=";", parse_dates=["data"])

cdi_serie = cdi_df.set_index("data")["cdi_diario"].reindex(ibov["data"]).ffill().bfill()
ibov["cdi_diario"] = cdi_serie.values
ibov["ibov_ret_log"] = np.log(1 + ibov["ibov_ret_simples"])

print(f"✓ Benchmark carregado e alinhado: {len(ibov):,} observações.")

✓ Benchmark carregado e alinhado: 3,966 observações.


## 3. Testes de Raiz Unitária e Ordem de Integração do IBOV

In [3]:
r = ibov["ibov_ret_log"].dropna()
p = ibov["ibov_close"]

adf_r = adfuller(r, autolag="AIC")
adf_p = adfuller(p, autolag="AIC")
kpss_r = kpss(r, regression="c", nlags="auto")
kpss_p = kpss(p, regression="c", nlags="auto")

print("========================================================================")
print("TESTES DE ESTACIONARIEDADE DO BENCHMARK (IBOV)")
print("========================================================================\n")
print(f"ADF em log-retornos:   estat={adf_r[0]:.4f} | p-valor={fmt_pvalor(adf_r[1])}")
print(f"KPSS em log-retornos:  estat={kpss_r[0]:.4f} | p-valor={fmt_pvalor(kpss_r[1])}")
print(f"ADF em nível (close):  estat={adf_p[0]:.4f} | p-valor={fmt_pvalor(adf_p[1])}")
print(f"KPSS em nível (close): estat={kpss_p[0]:.4f} | p-valor={fmt_pvalor(kpss_p[1])}")

TESTES DE ESTACIONARIEDADE DO BENCHMARK (IBOV)

ADF em log-retornos:   estat=-23.3832 | p-valor=< 0,001 ***
KPSS em log-retornos:  estat=0.1492 | p-valor=0.1000
ADF em nível (close):  estat=-0.0549 | p-valor=0.9537
KPSS em nível (close): estat=8.5742 | p-valor=0.0100 **


## 4. Diagnóstico de Momentos Superiores e Dependência Temporal

In [4]:
arch_stat, arch_p, _, _ = het_arch(r, nlags=10)
jb_stat, jb_p = stats.jarque_bera(r)

print("========================================================================")
print("DIAGNÓSTICO ESTATÍSTICO DE COMPORTAMENTO DISTRIBUTIVO (IBOV)")
print("========================================================================\n")
print(f"ARCH-LM (10 lags):  estat = {arch_stat:.2f}  p-valor = {fmt_pvalor(arch_p)}")
print(f'Jarque-Bera:        estat = {jb_stat:.2f}  p-valor = {fmt_pvalor(jb_p)}')
print(f"Assimetria:         {r.skew():+.4f}")
print(f"Curtose excedente:  {r.kurt():+.4f}")

DIAGNÓSTICO ESTATÍSTICO DE COMPORTAMENTO DISTRIBUTIVO (IBOV)

ARCH-LM (10 lags):  estat = 1514.10  p-valor = < 0,001 ***
Jarque-Bera:        estat = 24534.08  p-valor = < 0,001 ***
Assimetria:         -0.7859
Curtose excedente:  +12.0997


## 5. Teste de Razão de Variâncias de Lo-MacKinlay

In [5]:
log_price = np.log(p).values
vr_2 = VarianceRatio(log_price, 2, overlap=True)
vr_16 = VarianceRatio(log_price, 16, overlap=True)

print("========================================================================")
print("TESTE DE RAZÃO DE VARIÂNCIAS DE LO-MACKINLAY (1988) DO IBOV")
print("========================================================================\n")
print(f"Razão de Variâncias (k=2):   estat={vr_2.stat:.4f} | p-valor={fmt_pvalor(vr_2.pvalue)}")
print(f"Razão de Variâncias (k=16):  estat={vr_16.stat:.4f} | p-valor={fmt_pvalor(vr_16.pvalue)}")

TESTE DE RAZÃO DE VARIÂNCIAS DE LO-MACKINLAY (1988) DO IBOV

Razão de Variâncias (k=2):   estat=-1.5898 | p-valor=0.1119
Razão de Variâncias (k=16):  estat=-0.1700 | p-valor=0.8650


## 6. Modelagem de Volatilidade Condicional e Parâmetro Delta

In [6]:
r_pct = r * 100
res_garch = arch_model(r_pct, mean="Constant", vol="Garch", p=1, q=1, dist="normal").fit(disp="off")
res_gjr = arch_model(r_pct, mean="Constant", vol="Garch", p=1, o=1, q=1, dist="t").fit(disp="off")
res_eg = arch_model(r_pct, mean="Constant", vol="EGARCH", p=1, o=1, q=1, dist="t").fit(disp="off")

comp = pd.DataFrame([
    diagnostico_residuos(res_garch, "GARCH(1,1) Normal"),
    diagnostico_residuos(res_gjr,   "GJR-GARCH(1,1,1) t"),
    diagnostico_residuos(res_eg,    "EGARCH(1,1,1) t"),
])

melhor = comp.loc[comp["BIC"].idxmin(), "Modelo"]
res_melhor = {"GARCH(1,1) Normal": res_garch, "GJR-GARCH(1,1,1) t": res_gjr, "EGARCH(1,1,1) t": res_eg}[melhor]

print("========================================================================")
print("COMPARAÇÃO DE MODELOS DE VOLATILIDADE CONDICIONAL E CRITÉRIOS")
print("========================================================================\n")
print(comp.round(4).to_string(index=False))
print(f"\n>>> Modelo selecionado pelo BIC: {melhor}")

COMPARAÇÃO DE MODELOS DE VOLATILIDADE CONDICIONAL E CRITÉRIOS

            Modelo        AIC        BIC    Log-Lik  LB Q(10) em z  LB Q(10) em z²
 GARCH(1,1) Normal 13364.0813 13389.2234 -6678.0407         0.5968          0.6614
GJR-GARCH(1,1,1) t 13221.1679 13258.8810 -6604.5840         0.6162          0.9078
   EGARCH(1,1,1) t 13212.2077 13249.9208 -6600.1039         0.4278          0.2269

>>> Modelo selecionado pelo BIC: EGARCH(1,1,1) t


## 7. Consolidação de Estratégias e Análise de Sensibilidade

In [7]:
strat_pq = INPUT_DIR_STRATEGY / "strategy_returns.parquet"
strat_csv = INPUT_DIR_STRATEGY / "strategy_returns.csv"
if strat_pq.exists():
    strat = pd.read_parquet(strat_pq)
else:
    strat = pd.read_csv(strat_csv, index_col=0, parse_dates=True)

df = pd.DataFrame(index=strat.index)
df["ret_ibov"] = ibov.set_index("data")["ibov_ret_simples"].reindex(df.index).ffill().bfill()
df["rf"] = ibov.set_index("data")["cdi_diario"].reindex(df.index).ffill().bfill()

for est in ESTRATEGIAS:
    df[est] = strat[est]
    df[f"excess_{est}"] = df[est] - df["rf"]
df["excess_ibov"] = df["ret_ibov"] - df["rf"]

colunas_teste = ESTRATEGIAS + ["ret_ibov"]
linhas_perf = []
for c in colunas_teste:
    nm = "IBOVESPA" if c == "ret_ibov" else c
    r_serie = df[c]
    sh = sharpe(r_serie, df["rf"])
    so = sortino(r_serie, df["rf"])
    linhas_perf.append({
        "Estratégia": nm,
        "Retorno Acumulado (%)": float((1 + r_serie).prod() - 1) * 100,
        "CAGR (% a.a.)": cagr(r_serie) * 100,
        "Volatilidade (% a.a.)": float(r_serie.std() * np.sqrt(TRADING_DAYS)) * 100,
        "Sharpe": sh,
        "Sortino": so,
        "Max Drawdown (%)": max_drawdown(r_serie) * 100
    })
painel = pd.DataFrame(linhas_perf)

print("========================================================================")
print("PAINEL COMPARATIVO GERAL DE PERFORMANCE E RISCO")
print("========================================================================\n")
print(painel.round(4).to_string(index=False))

PAINEL COMPARATIVO GERAL DE PERFORMANCE E RISCO

         Estratégia  Retorno Acumulado (%)  CAGR (% a.a.)  Volatilidade (% a.a.)  Sharpe  Sortino  Max Drawdown (%)
        EqualWeight               301.6600        13.9120                19.5554  0.2902   0.4027          -35.5387
EqualWeight_BuyHold               448.3955        17.2839                19.2750  0.4429   0.6206          -32.9974
             InvVol               330.1981        14.6469                18.8964  0.3276   0.4554          -34.2814
             MinVar               268.6268        12.9999                12.9487  0.2931   0.4003          -25.0541
         MinVar_c10               274.7025        13.1731                12.9838  0.3044   0.4154          -24.9988
          MaxSharpe               303.6029        13.9636                17.5398  0.3046   0.4302          -22.7622
      MaxSharpe_c10               268.1330        12.9857                16.5432  0.2606   0.3649          -23.8932
           MaxOmega    

## 8. Diagnósticos por Estratégia — por que o bootstrap é necessário

Para cada uma das 15 estratégias: **Jarque-Bera** (normalidade),
**Ljung-Box(10)** (autocorrelação), **ARCH-LM(10)** (heterocedasticidade
condicional) e **ADF** (estacionariedade). A rejeição da normalidade na
maioria das séries invalida o teste paramétrico de Sharpe (Jobson-Korkie)
e justifica o bootstrap estacionário de Ledoit-Wolf (2008).


In [8]:
diag_estr = pd.DataFrame(
    {est: diagnosticos_serie(df[est].dropna(), ALPHA_SIG) for est in ESTRATEGIAS}
).T
n_nn = (diag_estr['normal?'] == 'não').sum()
print(f'Normalidade rejeitada em {n_nn}/{len(ESTRATEGIAS)} estratégias '
      f'(JB p<{ALPHA_SIG}) — bootstrap preferível ao teste paramétrico.\n')
print(diag_estr.round(4).to_string())


Normalidade rejeitada em 16/16 estratégias (JB p<0.05) — bootstrap preferível ao teste paramétrico.

                    assimetria   curtose JB_p LjungBox_p ARCH_p ADF_p normal?
EqualWeight          -0.354319  5.747073  0.0   0.003218    0.0   0.0     não
EqualWeight_BuyHold   -0.32162  5.825468  0.0   0.004674    0.0   0.0     não
InvVol               -0.352575  5.695393  0.0   0.003412    0.0   0.0     não
MinVar               -0.586794  6.899065  0.0   0.000017    0.0   0.0     não
MinVar_c10           -0.601624  7.000986  0.0   0.000015    0.0   0.0     não
MaxSharpe            -0.219078  4.868449  0.0    0.01349    0.0   0.0     não
MaxSharpe_c10        -0.298342  5.791288  0.0   0.013655    0.0   0.0     não
MaxOmega             -0.113458  5.219359  0.0   0.005772    0.0   0.0     não
MaxSortino           -0.202545  4.692766  0.0   0.011495    0.0   0.0     não
MaxKappa3            -0.168102  4.471118  0.0    0.00701    0.0   0.0     não
MinCVaR              -0.521385  6.365012 

## 9. Validação Não-Paramétrica via Stationary Bootstrap

In [9]:
print(f"Stationary bootstrap (B={BOOTSTRAP_REPS}, block_mean={BOOTSTRAP_BLOCK_MEAN}d)\n")
print(f"{'Métrica Estatística':<22} | {'Estratégia (IC 95%)':<36} | {'IBOVESPA (IC 95%)'}")
print("-" * 100)

for est in ESTRATEGIAS:
    print(f"\n========================================================================")
    print(f"INTERVALOS DE CONFIANÇA POR REAMOSTRAGEM: {est} VS IBOVESPA")
    print("========================================================================")
    for nome_fn, fn, serie_a, serie_b in [
        ("Sharpe (excesso)",  sharpe_de_excesso,  df[f"excess_{est}"], df["excess_ibov"]),
        ("Sortino (excesso)", sortino_de_excesso, df[f"excess_{est}"], df["excess_ibov"]),
        ("CAGR (bruto)",      cagr,               df[est],             df["ret_ibov"]),
    ]:
        s_lo, s_md, s_hi = bootstrap_ic(serie_a, fn, B=BOOTSTRAP_REPS, block_mean=BOOTSTRAP_BLOCK_MEAN, seed=SEED)  # unificado com seções 11-12
        i_lo, i_md, i_hi = bootstrap_ic(serie_b, fn, B=BOOTSTRAP_REPS, block_mean=BOOTSTRAP_BLOCK_MEAN, seed=SEED)  # unificado com seções 11-12
        print(f"{nome_fn:<22} | [{s_lo:+.4f},  {s_md:+.4f},  {s_hi:+.4f}]  | "
              f"[{i_lo:+.4f},  {i_md:+.4f},  {i_hi:+.4f}]")


Stationary bootstrap (B=2000, block_mean=10d)

Métrica Estatística    | Estratégia (IC 95%)                  | IBOVESPA (IC 95%)
----------------------------------------------------------------------------------------------------

INTERVALOS DE CONFIANÇA POR REAMOSTRAGEM: EqualWeight VS IBOVESPA


Sharpe (excesso)       | [-0.3576,  +0.3051,  +0.9955]  | [-0.3754,  +0.1931,  +0.8227]


Sortino (excesso)      | [-0.4804,  +0.4232,  +1.4324]  | [-0.4962,  +0.2672,  +1.2143]


CAGR (bruto)           | [+0.0032,  +0.1425,  +0.2971]  | [-0.0376,  +0.1156,  +0.2806]

INTERVALOS DE CONFIANÇA POR REAMOSTRAGEM: EqualWeight_BuyHold VS IBOVESPA


Sharpe (excesso)       | [-0.1800,  +0.4527,  +1.1258]  | [-0.3754,  +0.1931,  +0.8227]


...

Sortino (excesso)      | [-0.1426,  +0.7524,  +1.7693]  | [-0.4962,  +0.2672,  +1.2143]


CAGR (bruto)           | [+0.0102,  +0.2282,  +0.5035]  | [-0.0376,  +0.1156,  +0.2806]

INTERVALOS DE CONFIANÇA POR REAMOSTRAGEM: BL_downside_c10 VS IBOVESPA


Sharpe (excesso)       | [-0.1919,  +0.3949,  +1.0383]  | [-0.3754,  +0.1931,  +0.8227]


Sortino (excesso)      | [-0.2568,  +0.5570,  +1.5139]  | [-0.4962,  +0.2672,  +1.2143]


CAGR (bruto)           | [+0.0198,  +0.1689,  +0.3414]  | [-0.0376,  +0.1156,  +0.2806]


## 10. CAPM HAC, Jobson-Korkie/Memmel e Wald Spanning

In [10]:
X_capm = sm.add_constant((df["ret_ibov"] - df["rf"]).values)
X_span = sm.add_constant(df["ret_ibov"].values)
exc_ibov = df["ret_ibov"] - df["rf"]

linhas_testes = []
for est in ESTRATEGIAS:
    exc = df[est] - df["rf"]
    capm = sm.OLS(exc.values, X_capm).fit(cov_type="HAC", cov_kwds={"maxlags": SPANNING_MAX_LAGS})
    z_jk, p_jk = _jk_memmel(exc, exc_ibov)
    _, f_sp, p_sp = _wald_spanning(df[est].values, X_span, maxlags=SPANNING_MAX_LAGS)
    linhas_testes.append({
        "Estrategia":         est,
        "CAPM alfa (% a.a.)": capm.params[0] * TRADING_DAYS * 100,
        "CAPM beta":          capm.params[1],
        "CAPM t_alfa (NW)":   capm.tvalues[0],
        "CAPM p_alfa":        capm.pvalues[0],
        "JK/Memmel z":        z_jk,
        "JK/Memmel p":        p_jk,
        "Spanning F":         f_sp,
        "Spanning p":         p_sp,
        "Sharpe (rf var)":    sharpe(df[est], df["rf"]),
        "Sortino (rf var)":   sortino(df[est], df["rf"]),
    })

apendice_H = pd.DataFrame(linhas_testes).set_index("Estrategia")

print("========================================================================")
print("APÊNDICE L — INFERÊNCIA COMPARATIVA TRANSVERSAL (HAC Newey-West)")
print("========================================================================\n")
for est, row in apendice_H.iterrows():
    print(f"\n> {est}")
    print(f"   CAPM:     alfa = {row['CAPM alfa (% a.a.)']:+.2f}% a.a. | beta = {row['CAPM beta']:.3f} "
          f"| t(NW) = {row['CAPM t_alfa (NW)']:+.3f} | p = {fmt_pvalor(row['CAPM p_alfa'])}")
    print(f"   JK/Memmel vs IBOV: z = {row['JK/Memmel z']:+.3f} | p = {fmt_pvalor(row['JK/Memmel p'])}")
    print(f"   Spanning (a=0,b=1): F = {row['Spanning F']:.2f} | p = {fmt_pvalor(row['Spanning p'])}")
    print(f"   Sharpe = {row['Sharpe (rf var)']:+.4f} | Sortino = {row['Sortino (rf var)']:+.4f}")

APÊNDICE L — INFERÊNCIA COMPARATIVA TRANSVERSAL (HAC Newey-West)


> EqualWeight
   CAPM:     alfa = +2.49% a.a. | beta = 0.769 | t(NW) = +1.005 | p = 0.3148
   JK/Memmel vs IBOV: z = +0.900 | p = 0.3683
   Spanning (a=0,b=1): F = 20.74 | p = < 0,001 ***
   Sharpe = +0.2902 | Sortino = +0.4027

> EqualWeight_BuyHold
   CAPM:     alfa = +5.41% a.a. | beta = 0.755 | t(NW) = +2.236 | p = 0.0254 **
   JK/Memmel vs IBOV: z = +2.072 | p = 0.0382 **
   Spanning (a=0,b=1): F = 25.99 | p = < 0,001 ***
   Sharpe = +0.4429 | Sortino = +0.6206

> InvVol
   CAPM:     alfa = +3.10% a.a. | beta = 0.746 | t(NW) = +1.329 | p = 0.1839
   JK/Memmel vs IBOV: z = +1.231 | p = 0.2184
   Spanning (a=0,b=1): F = 25.77 | p = < 0,001 ***
   Sharpe = +0.3276 | Sortino = +0.4554

> MinVar
   CAPM:     alfa = +1.99% a.a. | beta = 0.436 | t(NW) = +0.831 | p = 0.4059
   JK/Memmel vs IBOV: z = +0.575 | p = 0.5650
   Spanning (a=0,b=1): F = 955.00 | p = < 0,001 ***
   Sharpe = +0.2931 | Sortino = +0.4003

> MinVar_c10

## 11. Testes de Diferença do Índice de Sharpe vs 1/N

H₀: SR_estratégia = SR_benchmark. Dois testes lado a lado:
- **Jobson-Korkie-Memmel (2003)** — paramétrico, baseline;
- **Ledoit-Wolf (2008)** — bootstrap estacionário, robusto a não-normalidade e autocorrelação.

P-valores ajustados por **Holm-Bonferroni** (15 comparações).
Benchmark: **EqualWeight** (1/N) — desafiante canônico de DeMiguel, Garlappi & Uppal (2009).


In [11]:
base_bench = df[BENCHMARK].dropna()
exc_bench  = (df[BENCHMARK] - df['rf']).dropna()

linhas_sr = []
for est in ESTRATEGIAS:
    if est == BENCHMARK:
        continue
    d = pd.concat([df[est], base_bench], axis=1).dropna()
    exc_est = (df[est]       - df['rf']).reindex(d.index)
    exc_bch = (df[BENCHMARK] - df['rf']).reindex(d.index)
    _, p_jk = _jk_memmel(exc_est, exc_bch)
    # LW bootstrap sobre retornos em EXCESSO — mesma base do JKM
    dsr, p_lw = lw_bootstrap_sharpe(
        exc_est.values, exc_bch.values,
        bloco=BOOTSTRAP_BLOCK_MEAN, reps=BOOTSTRAP_REPS, seed=SEED
    )
    linhas_sr.append({'estrategia': est, 'dSR_anual_vs_1N': dsr,
                      'JK_p': p_jk, 'LW_p': p_lw})

testes_sr = pd.DataFrame(linhas_sr).set_index('estrategia')
testes_sr['JK_p_holm'] = multipletests(testes_sr['JK_p'], method='holm')[1]
testes_sr['LW_p_holm'] = multipletests(testes_sr['LW_p'], method='holm')[1]
testes_sr['signif_LW'] = np.where(testes_sr['LW_p_holm'] < ALPHA_SIG, 'sim', 'nao')

print(f'Diferenca de Sharpe vs {BENCHMARK} (dSR anualizado sobre excesso; p Holm):\n')
print(testes_sr.round(4).to_string())


Diferenca de Sharpe vs EqualWeight (dSR anualizado sobre excesso; p Holm):

                     dSR_anual_vs_1N    JK_p    LW_p  JK_p_holm  LW_p_holm signif_LW
estrategia                                                                          
EqualWeight_BuyHold           0.1527  0.0388  0.0576     0.5433     0.7780       nao
InvVol                        0.0374  0.0203  0.0171     0.3049     0.2571       nao
MinVar                        0.0029  0.9860  0.9838     1.0000     1.0000       nao
MinVar_c10                    0.0142  0.9269  0.9157     1.0000     1.0000       nao
MaxSharpe                     0.0144  0.9395  0.9464     1.0000     1.0000       nao
MaxSharpe_c10                -0.0296  0.8619  0.8784     1.0000     1.0000       nao
MaxOmega                     -0.1415  0.4323  0.5014     1.0000     1.0000       nao
MaxSortino                   -0.0090  0.9633  0.9675     1.0000     1.0000       nao
MaxKappa3                    -0.0059  0.9768  0.9794     1.0000     1.0000

## 12. Testes de Diferença do Índice de Sortino vs 1/N

Mesmo procedimento de bootstrap aplicado ao Sortino — coerente com a PMPT,
por capturar exclusivamente o risco de queda. P-valores ajustados por Holm.


In [12]:
rf_medio = float(df['rf'].mean())

linhas_so = []
for est in ESTRATEGIAS:
    if est == BENCHMARK:
        continue
    d = pd.concat([df[est], base_bench], axis=1).dropna()
    exc_est_so = (df[est]       - df['rf']).reindex(d.index).values
    exc_bch_so = (df[BENCHMARK] - df['rf']).reindex(d.index).values
    # LW bootstrap sobre excesso com rf=0.0 (rf ja subtraido nos inputs)
    dso, p = lw_bootstrap_sortino(
        exc_est_so, exc_bch_so,
        rf=0.0, bloco=BOOTSTRAP_BLOCK_MEAN, reps=BOOTSTRAP_REPS, seed=SEED
    )
    linhas_so.append({'estrategia': est, 'dSortino_anual': dso, 'LW_p': p})

testes_so = pd.DataFrame(linhas_so).set_index('estrategia')
testes_so['LW_p_holm'] = multipletests(testes_so['LW_p'], method='holm')[1]
testes_so['signif_LW'] = np.where(testes_so['LW_p_holm'] < ALPHA_SIG, 'sim', 'nao')

print(f'Diferenca de Sortino vs {BENCHMARK} (bootstrap excesso rf=0; p Holm):\n')
print(testes_so.round(4).to_string())


Diferenca de Sortino vs EqualWeight (bootstrap excesso rf=0; p Holm):

                     dSortino_anual    LW_p  LW_p_holm signif_LW
estrategia                                                      
EqualWeight_BuyHold          0.2179  0.0554     0.7752       nao
InvVol                       0.0527  0.0176     0.2636       nao
MinVar                      -0.0024  0.9904     1.0000       nao
MinVar_c10                   0.0127  0.9462     1.0000       nao
MaxSharpe                    0.0274  0.9281     1.0000       nao
MaxSharpe_c10               -0.0378  0.8905     1.0000       nao
MaxOmega                    -0.1924  0.5194     1.0000       nao
MaxSortino                  -0.0054  0.9861     1.0000       nao
MaxKappa3                    0.0002  0.9995     1.0000       nao
MinCVaR                     -0.0401  0.8602     1.0000       nao
MinCDaR                     -0.5481  0.0588     0.7752       nao
BL_classico                  0.4875  0.2396     1.0000       nao
BL_classico_c10    

## 11b–12b. Testes de Diferenca de Sharpe e Sortino vs Tres Benchmarks

Mesmo protocolo das secoes 11–12, repetido para tres benchmarks simultaneos:

1. **EqualWeight** — 1/N rebalanceado (DeMiguel, Garlappi & Uppal, 2009);
2. **EW_BuyHold** — 1/N comprar-e-manter (sem rebalanceamento);
3. **IBOV** — IBOVESPA (indice de mercado brasileiro).

**Base de calculo:** todos os testes (JKM e LW bootstrap) operam sobre retornos em **excesso** (serie - CDI diario). Para o Sortino, os inputs ja chegam descontados de rf, portanto `rf=0.0` na chamada interna.

**Correcao de Holm** aplicada **separadamente** dentro de cada benchmark (a familia de testes e o conjunto de estrategias comparadas aquele benchmark).

**Nota metodologica (Sharpe):** `lw_bootstrap_sharpe` calcula internamente `mean(x)/std(x)` sem parametro rf proprio. Passar retornos em excesso garante que LW e JKM operem na mesma base — alinhamento que as secoes 11 e 12 originais tambem passaram a adotar nesta revisao.

**Impacto empirico da mudanca de base (seed=42, B=2000):**

| Par | dSR bruto | dSR excesso | diferenca |
|-----|-----------|-------------|-----------|
| EqualWeight_BuyHold vs EqualWeight | 0.4610 | 0.4478 | +0.013 |
| MinVar vs EqualWeight | 0.5425 | 0.2942 | +0.248 |
| BL_classico vs EqualWeight | 0.4739 | 0.5850 | -0.111 |

O desvio e material para estrategias com volatilidade muito diferente do benchmark (MinVar, BL). Para o Sortino, o impacto foi inferior a 0.001 em todos os casos (rf_medio como escalar constante ja era uma boa aproximacao).

Resultados exportados em `inferencia_sharpe_3benchmarks.csv` e `inferencia_sortino_3benchmarks.csv`.

In [13]:
BENCHMARKS_3 = {
    "EqualWeight":    "EqualWeight",
    "EW_BuyHold":     "EqualWeight_BuyHold",
    "IBOV":           "ret_ibov",
}

# ── Sharpe triplo ──────────────────────────────────────────────────────────
all_sr3 = []
for label, col_bench in BENCHMARKS_3.items():
    linhas = []
    for est in ESTRATEGIAS:
        if est == col_bench:
            continue
        d = pd.concat([df[est], df[col_bench]], axis=1).dropna()
        exc_est = (df[est]       - df["rf"]).reindex(d.index)
        exc_bch = (df[col_bench] - df["rf"]).reindex(d.index)
        _, p_jk  = _jk_memmel(exc_est, exc_bch)
        dsr, p_lw = lw_bootstrap_sharpe(
            exc_est.values, exc_bch.values,
            bloco=BOOTSTRAP_BLOCK_MEAN, reps=BOOTSTRAP_REPS, seed=SEED
        )
        linhas.append({"estrategia": est, "benchmark": label,
                       "dSR_anual": dsr, "JK_p": p_jk, "LW_p": p_lw})
    tmp = pd.DataFrame(linhas)
    tmp["LW_p_holm"] = multipletests(tmp["LW_p"], method="holm")[1]
    tmp["signif_LW"] = np.where(tmp["LW_p_holm"] < ALPHA_SIG, "sim", "nao")
    all_sr3.append(tmp)

sr3 = pd.concat(all_sr3, ignore_index=True)

sr_wide = sr3.pivot(index="estrategia", columns="benchmark",
                    values=["dSR_anual", "LW_p", "LW_p_holm", "signif_LW"])
sr_wide.columns = [f"{v}|{b}" for v, b in sr_wide.columns]
sr_wide = sr_wide.reindex(ESTRATEGIAS)

print("=" * 80)
print("DIFERENCA DE SHARPE vs TRES BENCHMARKS (dSR excesso anualizado | p Holm)")
print("=" * 80)
for bench_label in BENCHMARKS_3:
    cols = [c for c in sr_wide.columns if c.endswith(f"|{bench_label}")]
    sub = sr_wide[cols].dropna()
    if not sub.empty:
        sub.columns = [c.split("|")[0] for c in cols]
        print(f"\n-- Benchmark: {bench_label} --")
        print(sub.round(4).to_string())

sr3.to_csv(OUTPUT_DIR / "inferencia_sharpe_3benchmarks.csv", index=False, float_format="%.6f")
print(f"\nExportado: inferencia_sharpe_3benchmarks.csv")

# ── Sortino triplo ─────────────────────────────────────────────────────────
all_so3 = []
for label, col_bench in BENCHMARKS_3.items():
    linhas = []
    for est in ESTRATEGIAS:
        if est == col_bench:
            continue
        d = pd.concat([df[est], df[col_bench]], axis=1).dropna()
        exc_est_so = (df[est]       - df["rf"]).reindex(d.index).values
        exc_bch_so = (df[col_bench] - df["rf"]).reindex(d.index).values
        dso, p = lw_bootstrap_sortino(
            exc_est_so, exc_bch_so,
            rf=0.0, bloco=BOOTSTRAP_BLOCK_MEAN, reps=BOOTSTRAP_REPS, seed=SEED
        )
        linhas.append({"estrategia": est, "benchmark": label,
                       "dSortino_anual": dso, "LW_p": p})
    tmp = pd.DataFrame(linhas)
    tmp["LW_p_holm"] = multipletests(tmp["LW_p"], method="holm")[1]
    tmp["signif_LW"] = np.where(tmp["LW_p_holm"] < ALPHA_SIG, "sim", "nao")
    all_so3.append(tmp)

so3 = pd.concat(all_so3, ignore_index=True)

so_wide = so3.pivot(index="estrategia", columns="benchmark",
                    values=["dSortino_anual", "LW_p", "LW_p_holm", "signif_LW"])
so_wide.columns = [f"{v}|{b}" for v, b in so_wide.columns]
so_wide = so_wide.reindex(ESTRATEGIAS)

print("\n" + "=" * 80)
print("DIFERENCA DE SORTINO vs TRES BENCHMARKS (excesso rf=0 | p Holm)")
print("=" * 80)
for bench_label in BENCHMARKS_3:
    cols = [c for c in so_wide.columns if c.endswith(f"|{bench_label}")]
    sub = so_wide[cols].dropna()
    if not sub.empty:
        sub.columns = [c.split("|")[0] for c in cols]
        print(f"\n-- Benchmark: {bench_label} --")
        print(sub.round(4).to_string())

so3.to_csv(OUTPUT_DIR / "inferencia_sortino_3benchmarks.csv", index=False, float_format="%.6f")
print(f"\nExportado: inferencia_sortino_3benchmarks.csv")


DIFERENCA DE SHARPE vs TRES BENCHMARKS (dSR excesso anualizado | p Holm)

-- Benchmark: EqualWeight --
                    dSR_anual      LW_p LW_p_holm signif_LW
estrategia                                                 
EqualWeight_BuyHold  0.152712  0.057569  0.778008       nao
InvVol               0.037431  0.017142  0.257123       nao
MinVar                0.00287  0.983802       1.0       nao
MinVar_c10           0.014233  0.915738       1.0       nao
MaxSharpe            0.014414  0.946367       1.0       nao
MaxSharpe_c10       -0.029604  0.878392       1.0       nao
MaxOmega            -0.141454  0.501399       1.0       nao
MaxSortino          -0.008954   0.96749       1.0       nao
MaxKappa3           -0.005932  0.979449       1.0       nao
MinCVaR             -0.025808  0.873262       1.0       nao
MinCDaR             -0.395212  0.055572  0.778008       nao
BL_classico          0.320782  0.259824       1.0       nao
BL_classico_c10      0.273044  0.190409       1.0       n


DIFERENCA DE SORTINO vs TRES BENCHMARKS (excesso rf=0 | p Holm)

-- Benchmark: EqualWeight --
                    dSortino_anual      LW_p LW_p_holm signif_LW
estrategia                                                      
EqualWeight_BuyHold       0.217885  0.055372  0.775206       nao
InvVol                    0.052662  0.017572  0.263577       nao
MinVar                   -0.002392  0.990373       1.0       nao
MinVar_c10                 0.01272  0.946205       1.0       nao
MaxSharpe                 0.027446  0.928136       1.0       nao
MaxSharpe_c10            -0.037789  0.890517       1.0       nao
MaxOmega                 -0.192366  0.519388       1.0       nao
MaxSortino               -0.005441  0.986088       1.0       nao
MaxKappa3                 0.000199  0.999515       1.0       nao
MinCVaR                  -0.040066  0.860204       1.0       nao
MinCDaR                  -0.548087  0.058798  0.775206       nao
BL_classico                 0.4875  0.239557       1.0      

## 13. Exportação dos Resultados e Apêndices

In [14]:
apendice_G = pd.DataFrame([
    {"Teste": "ADF (log-retornos)",     "H₀": "raiz unitária",    "Estat.": adf_r[0],  "p-valor": adf_r[1]},
    {"Teste": "KPSS (log-retornos)",    "H₀": "estacionariedade", "Estat.": kpss_r[0], "p-valor": kpss_r[1]},
    {"Teste": "ADF (nível)",            "H₀": "raiz unitária",    "Estat.": adf_p[0],  "p-valor": adf_p[1]},
    {"Teste": "KPSS (nível)",           "H₀": "estacionariedade", "Estat.": kpss_p[0], "p-valor": kpss_p[1]},
    {"Teste": "ARCH-LM (10 lags)",      "H₀": "variância constante", "Estat.": arch_stat, "p-valor": arch_p},
    {"Teste": "Jarque-Bera",            "H₀": "normalidade",      "Estat.": jb_stat,        "p-valor": jb_p},
    {"Teste": "VR k=2 (Lo-MacKinlay)",  "H₀": "passeio aleatório", "Estat.": vr_2.stat,  "p-valor": vr_2.pvalue},
    {"Teste": "VR k=16 (Lo-MacKinlay)", "H₀": "passeio aleatório", "Estat.": vr_16.stat, "p-valor": vr_16.pvalue},
])
apendice_G["Decisão"] = np.where(apendice_G["p-valor"] < 0.05, "Rejeita H₀ ✗", "Não rejeita H₀ ✓")
apendice_G.to_csv(OUTPUT_DIR / "apendice_G_diagnostico_ibov.csv", index=False)

apendice_H.to_csv(OUTPUT_DIR / "apendice_H_testes_estrategia.csv", float_format="%.6f")
painel.to_csv(OUTPUT_DIR / "apendice_H_painel_metricas.csv", index=False)

comp_rf_linhas = []
for c in colunas_teste:
    nm = "IBOVESPA" if c == "ret_ibov" else c
    comp_rf_linhas.append({
        "Estratégia": nm,
        "Sharpe (CDI)": sharpe(df[c], df["rf"]),
        "Sharpe (Zero)": sharpe(df[c], 0.0),
        "Sortino (CDI)": sortino(df[c], df["rf"]),
        "Sortino (Zero)": sortino(df[c], 0.0),
    })
comp_rf = pd.DataFrame(comp_rf_linhas)
comp_rf.to_csv(OUTPUT_DIR / "apendice_H_comparativo_rf.csv", index=False)

# Diagnósticos e testes de diferença (do NB07 atualizado)
diag_estr.to_csv(OUTPUT_DIR / 'inferencia_diagnosticos.csv', float_format='%.6f')
testes_sr.to_csv(OUTPUT_DIR / 'inferencia_sharpe_testes.csv', float_format='%.6f')
testes_so.to_csv(OUTPUT_DIR / 'inferencia_sortino_testes.csv', float_format='%.6f')

print('✓ CSVs de apêndice exportados com sucesso em:', OUTPUT_DIR)
print('  + inferencia_diagnosticos.csv')
print('  + inferencia_sharpe_testes.csv')
print('  + inferencia_sortino_testes.csv')

✓ CSVs de apêndice exportados com sucesso em: C:\VSCodeWorkspace\1_TCC_Final\data\Estrategias
  + inferencia_diagnosticos.csv
  + inferencia_sharpe_testes.csv
  + inferencia_sortino_testes.csv


# Autoavaliação — *Ten Simple Rules* (Rule et al., 2019)

> Rule A, Birmingham A, Zuniga C, Altintas I, Huang S-C, Knight R, Moshiri N, Nguyen MH,
> Rosenthal SB, Pérez F, Rose PW. *Ten simple rules for writing and sharing computational analyses
> in Jupyter Notebooks.* PLOS Comput Biol. 2019;15(7):e1007007.

| Regra | Tema | Status | Evidência / Aplicação no NB 09_01 (Inferência Econométrica) |
| :---: | :--- | :---: | :--- |
| 1 | Contar uma história | ✅ | Narrativa estruturada com introdução metodológica, células de cálculo comentadas e interpretação dos outputs. |
| 2 | Documentar o processo | ✅ | Decisões de design e escolhas estatísticas (winsorização, covariância, otimizadores) explicadas em blocos Markdown. |
| 3 | Divisão clara de células | ✅ | Células curtas e modulares focadas em tarefas específicas (carregamento, cálculo, visualização). |
| 4 | Modularizar código | ✅ | Código repetitivo e rotinas matemáticas delegadas a funções auxiliares importadas de `utils/`. |
| 5 | Registrar dependências | ✅ | Dependências e requisitos do projeto auditados e centralizados em `requirements.txt` e `requirements.py`. |
| 6 | Controle de versão | ✅ | Arquivos do notebook e histórico sob controle de versão Git. |
| 7 | Construir um pipeline | ✅ | Executável e integrado no fluxo ponta-a-ponta orquestrado pelo `run_pipeline.py`. |
| 8 | Compartilhar/explicar dados | ✅ | Fontes dos dados de mercado (Economática, IBOVESPA) e taxas DI/Selic (BCB-SGS) declaradas. |
| 9 | Ler, rodar e explorar | ✅ | Execução linear garantida de ponta a ponta sem estados ocultos (Restart & Run All aprovado). |
| 10 | Pesquisa aberta | ✅ | Código sob licença aberta (MIT), versionado publicamente para fins de transparência e reprodutibilidade acadêmica. |
